In [1]:
# Import standard libraries
from dataclasses import replace
import os
from pathlib import Path
import sys
import time

# Third-party libraries
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
import numpy as np
import torch

In [2]:
# Add the folder containing our envs/ and rl/ packages to the path
sys.path.append("/workspace/software")

# Import the custom environment and PPO training module
from envs.balance_bot_env import BalanceBotEnv
from rl.ppo_trainer import PPOConfig, evaluate, train, export_tb_plots_as_csv

In [3]:
# Settings
MJCF_PATH = Path("/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml")
SEED = 42
NUM_ENVS = 4               # Number of parallel environments. Only the first will be rendered.
STEPS_PER_ENV = 1_000_000    # Number of simulation steps to perform per environment

In [4]:
# Configure PPO
ppo_config = PPOConfig(
    exp_name = "balance-bot-ppo",  # Name of the experiment
    env_id = "BalanceBot-v0",      # Name of the environment
    seed = SEED,                   # Constant seed for reproducibility
    num_envs = NUM_ENVS,           # Number of parallel environments
    total_timesteps = NUM_ENVS * STEPS_PER_ENV,  # Total simulation steps (all envs and iterations)
    num_steps = 2048,              # Number of steps per rollout per env (2048 * 0.002s = ~4 sec)
    num_minibatches = 32,          # Number of minibatches for each training epoch
    update_epochs = 10,            # Number of epochs to update actor and critic for each iteration
    anneal_lr = True,              # Enable annealing (lower learning rate as training goes on)
    learning_rate = 3e-4,          # Initial learning rate, reduced by annealing (if enabled)
    gamma = 0.99,                  # Discount factor (future rewards are discounted by this amount)
    gae_lambda = 0.95,             # GAE blending: 0 = pure TD error, 1 = pure Monte Carlo
    clip_coef = 0.2,               # Limits policy ratio to prevent large actor updates
    value_clip = 1.0,              # Absolute bounds on value prediction change per update (critic)
    ent_coef = 0.0,                # How much entropy factors into total loss calculation
    vf_coef = 0.5,                 # How much the value loss factors into total loss calculation
    max_grad_norm = 0.5,           # Limits how much actor/critic parameters can change during an update
    checkpoint_interval = 50,      # Save model every 50 iterations
    save_model = True,             # Save the final model
    timestep = 0.000,              # Match MJCF opt.timestep for real-time rendering (or 0 for fast)
)

In [5]:
def make_balance_bot_env(render):
    """Function to create an environment for our balance bot"""
    # Create the environment and set the render mode
    env = BalanceBotEnv(
        mjcf_path    = MJCF_PATH,
        render_mode  = "human" if render else None,
    )

    # Wrap in RecordEpisodeStatistics so we can log episodic returns in the 'info' dict
    return gym.wrappers.RecordEpisodeStatistics(env)

In [6]:
# Build list of environment factory functions
env_factories = []
for i in range(NUM_ENVS):
    env_factories.append(lambda render=(i==0): make_balance_bot_env(render))

# Build vector of environments
envs = gym.vector.SyncVectorEnv(env_factories)

In [7]:
# Choo choo train!
result = train(ppo_config, envs=envs)

Run name: BalanceBot-v0__balance-bot-ppo__42__1777844769
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-ppo
Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1777844769/checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=4078.78) to runs/BalanceBot-v0__balance-bot-ppo__42__1777844769/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1777844769/checkpoint_iter0100.cleanrl_model
New best model saved (mean_return=8838.71) to runs/BalanceBot-v0__balance-bot-ppo__42__1777844769/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1777844769/checkpoint_iter0150.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1777844769/checkpoint_iter0200.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1777844769/checkpoint_iter0250.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1777844769/checkpoint_iter0300.cle

In [8]:
# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Best model: runs/BalanceBot-v0__balance-bot-ppo__42__1777844769/best_model.cleanrl_model
Final model: runs/BalanceBot-v0__balance-bot-ppo__42__1777844769/balance-bot-ppo_final.cleanrl_model
Best mean return: 8838.71
Loaded best model (mean_return=8838.71)


In [9]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.002)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

Mean return: 1970.77


In [10]:
# Close the environments
envs.close()

In [11]:
# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

I0000 00:00:1777849698.411771  168699 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777849698.443590  168699 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777849699.803233  168699 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/config/.config/matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /tmp/matplotlib-np

Exported runs/BalanceBot-v0__balance-bot-ppo__42__1777844769/charts_metrics.csv (5 metrics, 1883 steps)
Exported runs/BalanceBot-v0__balance-bot-ppo__42__1777844769/losses_metrics.csv (7 metrics, 488 steps)
